# 7주차 워크북 2
제공 PDF는 Google Drive의 [전체 ZIP](https://drive.google.com/file/d/1SYb8OH0cPCH8kjgq3arDDYvoWm_gB85f/view)에서 `Week07/data/week07_course_guide.pdf`를 받아 Colab에 업로드하세요.


In [ ]:
!pip -q install pypdf scikit-learn gradio openai


In [ ]:
from pathlib import Path
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
file = Path("week07_course_guide.pdf")
assert file.is_file(), "먼저 PDF를 Colab에 업로드하세요."
reader = PdfReader(str(file))
records = [(p, (page.extract_text() or "").strip())
           for p, page in enumerate(reader.pages, start=1)]
records = [(p, t) for p, t in records if t]
assert records, "스캔 PDF에서는 별도의 OCR이 필요합니다."
print("추출 페이지 수:", len(records))



In [ ]:
def chunk_words(text, size=75, overlap=20):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("0 <= overlap < size 조건이 필요합니다.")
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size])
            for i in range(0, len(words), step)
            if words[i:i + size]]

chunks = [{"page": p, "text": part}
          for p, text in records
          for part in chunk_words(text)]
print("청크 수:", len(chunks))



In [ ]:
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
vectors = vectorizer.fit_transform([x["text"] for x in chunks])

def retrieve(question, top_k=3):
    query = vectorizer.transform([question])
    scores = cosine_similarity(query, vectors).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return [{**chunks[int(i)], "score": float(scores[int(i)])}
            for i in order]

for question in ["RAG가 무엇인가요?", "API 키는 어떻게 관리하나요?",
                 "청크를 왜 만드나요?"]:
    best = retrieve(question, 1)[0]
    print(question, "페이지:", best["page"],
          "유사도:", round(best["score"], 3))



## Gradio UI + 선택형 LLM
Colab Secrets에 OPENAI_API_KEY를 등록하면 선택적으로 LLM 답변을 사용할 수 있습니다. 요금 발생 가능.


In [ ]:
import os
import gradio as gr
try:
    from google.colab import userdata
    secret = userdata.get("OPENAI_API_KEY")
    if secret:
        os.environ["OPENAI_API_KEY"] = secret
except Exception:
    print("API 키가 없으면 무료 모드로 진행합니다.")

def answer(question, use_llm=False):
    if not (question or "").strip():
        return "질문을 입력하세요.", ""
    hits = retrieve(question)
    evidence = "\n\n".join(
        f"PDF p.{x['page']} / {x['score']:.3f}\n{x['text']}"
        for x in hits
    )
    if hits[0]["score"] < 0.03:
        return "문서에서 관련 근거를 확인할 수 없습니다.", evidence
    if not use_llm:
        return "무료 검색 모드입니다. 오른쪽 근거를 확인하세요.", evidence
    key = os.environ.get("OPENAI_API_KEY", "")
    if not key:
        return "API 키가 없어 무료 검색 모드로 진행합니다.", evidence
    from openai import OpenAI
    try:
        response = OpenAI(api_key=key).responses.create(
            model="gpt-4.1-mini",
            instructions="제시된 근거만 사용해 한국어로 답하고 페이지를 표시하세요. 근거가 없으면 모른다고 하세요.",
            input=f"질문: {question}\n근거:\n{evidence}"
        )
        return response.output_text, evidence
    except Exception as e:
        return f"API 호출 오류: {type(e).__name__}", evidence

app = gr.Interface(
    fn=answer,
    inputs=[
        gr.Textbox(label="문서 질문"),
        gr.Checkbox(label="LLM 답변 사용 (선택)", value=False)
    ],
    outputs=[
        gr.Textbox(label="답변"),
        gr.Textbox(label="검색 근거", lines=12)
    ],
    title="PDF RAG 챗봇"
)
app.launch(share=False)



## 학생 과제
질문 세 개의 근거 페이지와 점수를 기록합니다. 청크 크기 변경 후 비교하고, 워크북 2는 UI 화면을 캡처하세요.
핵심 코드: PdfReader, extract_text, TfidfVectorizer, cosine_similarity, gr.Interface.
